In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### Import CSV

In [ ]:
df_index = pd.read_csv('./data/economic_index.csv')
df_index.head()

### Remove unnecessary columns

In [ ]:
# axis=1 specifies that we want to drop columns (not rows)
# inplace=True means the changes are applied directly to df_index, without needing to assign the result to a new variable
df_index.drop(columns=['Unnamed: 0', 'year', 'month'], axis=1, inplace=True)
df_index.head()

### To check null values

In [ ]:
df_index.isnull().sum()

### Visualize the data

In [ ]:
import seaborn as sns

sns.pairplot(df_index)

### To check correlation between the variables

In [ ]:
# Lets check the correlation between the variables

df_index.corr()

### Get independent and dependent features

In [ ]:
X = df_index[['interest_rate', 'unemployment_rate']]
y = df_index['unemployment_rate']

# OR

# Get all columns except the last one
X = df_index.iloc[:, :-1]
# Get just the last column
y = df_index.iloc[:, -1]

print(X.head())
print(y.head())

### Train test split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


### Standardize the data

In [ ]:
# We use StandardScaler to scale the features so they have mean 0 and standard deviation 1.
# This helps many machine learning models work better and train faster.
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### Train model

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression(n_jobs=-1)
model.fit(X_train, y_train)

### Perform cross validation

In [ ]:
# Cross-validation helps check how well the model works on new, unseen data and reduces the chance of overfitting.
from sklearn.model_selection import cross_val_score

# scoring='neg_mean_squared_error' uses negative mean squared error for evaluation, common in regression. 
# cv=3 means 3-fold cross-validation.
scores = cross_val_score(model, X_train, y_train, scoring='neg_mean_squared_error', cv=3)
print("Cross-validation scores:", scores)
print("Mean cross-validation score:", scores.mean())

### Error analysis

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

y_predict = model.predict(X_test)

mse = mean_squared_error(y_test, y_predict)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_predict)

print("Mean Squared Error:", mse)
print("Root Mean Squared Error:", rmse)
print("Mean Absolute Error:", mae)

# Since my target is index_price (in index points), 
# my RMSE≈83 and MAE≈72 mean the model’s predictions are typically off by about 70–80 index points on average.

### Performance Metrics

In [ ]:
from sklearn.metrics import r2_score

# R-squared (R²): Proportion of variance in the target explained by the model (ranges from 0 to 1; higher is better)
r2_squared = r2_score(y_test, y_predict)

# Adjusted R-squared: Adjusts R² for the number of predictors and data points
n = X_test.shape[0]  # number of samples
p = X_test.shape[1]  # number of predictors
adjusted_r2 = 1 - (1 - r2_squared) * (n - 1) / (n - p - 1)

print("R-squared (R²):", r2_squared)
print("Adjusted R-squared:", adjusted_r2)

### Assumptions

In [ ]:
# If we can see some patterns after scatter plot, then we can say the model performing well
plt.scatter(y_test, y_predict)

In [ ]:
# If we see a normal distribution of residuals, then we can say the model is performing well
residuals = y_test - y_predict
sns.displot(residuals, kind='kde')

In [ ]:
# If we see data is uniformly distributed, then we can say the model is performing well
plt.scatter(y_predict, residuals)